<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Why accuracy lies here

**[Why accuracy lies here](https://learning.nextia-ai.com/courses/anomaly-detection/m07/why-accuracy-lies-here/)** · Anomaly Detection: Find What Is Unusual · Module 7, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see why accuracy says almost nothing when anomalies are rare, and measure a detector the way the business uses it: precision at k, recall of known cases, the precision–recall curve and the time to detection.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | Precision, recall and the PR curve, from the machine learning course; Isolation Forest, from [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m07/why-accuracy-lies-here/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M07-L01-why-accuracy-lies-here/why-accuracy-lies-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. 99.9% accurate, and useless

A smoke alarm that never rings is right on every day without a fire: almost every day. It is still useless.

Run the next cell: 1,000 payments, 1 fraud, and a "detector" that never alerts.

> **Check.** You should see an accuracy of 0.999 and 0 of 1 frauds found.

In [ ]:
is_fraud = np.zeros(1000, dtype=bool)
is_fraud[417] = True                                  # 1 fraud in 1,000 payments
says_nothing = np.zeros(1000, dtype=bool)             # a "detector" that never alerts
accuracy = (says_nothing == is_fraud).mean()
print(f"accuracy of 'nothing is fraud': {accuracy:.3f}; frauds found: {int((says_nothing & is_fraud).sum())} of 1")

**Accuracy** counts the right answers among all answers. When 999 answers are "normal" and right, it is 99.9% whatever the detector does with the one that matters.

## 2. On Larkfield's payments

This lesson **reveals** the answer key on purpose: `compare.planted_payments()` says which evaluation payments are planted. The detector is Isolation Forest on the behaviour features (Module 4).

> **Check.** You should see 169,803 evaluation payments and 2,817 planted (1.66%).

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(scaler.transform(behaviour.loc[reference.index]))
score = -forest.score_samples(scaler.transform(behaviour.loc[evaluation.index]))
planted = evaluation.payment_id.isin(compare.planted_payments().payment_id).to_numpy()   # the reveal, for this lesson
print(f"{len(evaluation):,} evaluation payments, {planted.sum():,} planted ({planted.mean():.2%})")

> **Predict.** Isolation Forest alerts on 50 payments a day and finds most planted frauds. "Nothing is fraud" finds none. How far apart will their accuracies be?

In [ ]:
def accuracy_at_budget(score, budget=50):
    alerts = compare.daily_alerts(evaluation, score, budget)
    flagged = alerts.set_index("payment_id").reviewed.reindex(evaluation.payment_id).to_numpy()
    return (flagged == planted).mean()

print(f"nothing is fraud:        accuracy {(~planted).mean():.4f}")
print(f"Isolation Forest, 50/day: accuracy {accuracy_at_budget(score):.4f}")
print(f"perfect detector, 50/day: accuracy {accuracy_at_budget(planted + 0.0):.4f}")

> **Check.** You should see 0.9834 for "nothing is fraud", 0.9834 for Isolation Forest and 0.9857 for a perfect detector.

The three accuracies are within 0.2 points of each other. Accuracy cannot tell a useless detector from a perfect one here.

## 3. Precision at k

A team does not read every score: it reads the top **k** alerts each day. **Precision at k** is the share of those k that are real; **recall** is the share of all real cases that they find. Run the next cell for k from 10 to 200 alerts a day.

In [ ]:
rows = []
for k in (10, 25, 50, 100, 200):
    r = compare.score_payments(evaluation, score, "Isolation Forest", f"top {k} a day", budget=k)
    rows.append({"alerts a day (k)": k, "precision at k": r["precision at budget"], "recall": r["recall"]})
pd.DataFrame(rows)

> **Check.** At k = 50 you should see a precision of 0.500 and a recall of 0.808.

More alerts a day find more frauds (recall rises) but each alert is less often real (precision falls). Where to stop is a decision about the team's time: the next lesson, *The alert budget*.

## 4. The precision–recall curve

The machine learning course drew the **PR curve** for a classifier. The same curve works for anomaly scores: every threshold gives one precision and one recall. Its area, the **average precision**, summarises it in one number; random scores give the share of planted payments.

In [ ]:
precision, recall, _ = precision_recall_curve(planted, score)
ap = average_precision_score(planted, score)
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.plot(recall, precision)
ax.axhline(planted.mean(), color="grey", ls="--", label=f"random ({planted.mean():.3f})")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title(f"Isolation Forest, average precision {ap:.3f}")
ax.legend()
plt.show()

## 5. Time to detection

For fraud, finding it **early** matters: each minute, a stolen card pays again. Run the next cell. For each planted episode (all the payments of one fraud), it measures the minutes from the first payment to the first alert.

In [ ]:
episodes = compare.planted_payments().set_index("payment_id")
alerts = compare.daily_alerts(evaluation, score, 50)
a = alerts[alerts.payment_id.isin(episodes.index)].copy()
a["episode"], a["kind"] = a.payment_id.map(episodes.episode), a.payment_id.map(episodes.kind)
first = a.groupby("episode").agg(kind=("kind", "first"), start=("ts", "min"),
                                 first_alert=("ts", lambda s: s[a.loc[s.index, "reviewed"]].min()))
first["minutes"] = (first.first_alert - first.start).dt.total_seconds() / 60
first.groupby("kind").agg(episodes=("minutes", "size"), found=("minutes", "count"), median_minutes=("minutes", "median"))

> **Check.** The median minutes to the first alert are account_takeover 0.0, card_testing 0.8, stolen_card 0.0.

A detector can have a good precision and still alert after the money is gone.

## 6. Check yourself

**Your turn 1.** What is the accuracy of "nothing is fraud" on the evaluation payments? Write it as `accuracy_nothing`, rounded to 4 decimals.

In [ ]:
accuracy_nothing = ...   # your code

In [ ]:
expect('accuracy_nothing', accuracy_nothing, 0.9834, tolerance=0.0001)

**Your turn 2.** What is Isolation Forest's precision at k = 100 alerts a day? Write it as `precision_at_100`.

In [ ]:
precision_at_100 = ...   # your code

In [ ]:
expect_hash('precision_at_100', precision_at_100, 'f972b281cf77c946')

## Recap

- With rare anomalies, accuracy is about the same for every detector (0.9834 for one that does nothing).
- Precision at k and recall describe what the team sees and what it misses; the PR curve shows every k at once.
- Time to detection says if the alert comes in time.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m07/why-accuracy-lies-here/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.